In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm import tqdm

# Loading clusters, measurements, events
clusters_df = pd.read_csv('data/clinical_report_clean.csv')
measurements = pd.read_csv('data/2025_04_15/Первичные данные от 10.04.2025.csv', 
                           usecols=['id пациента', 'время измерения', 'САД', 'ДАД', 'ЧП'])
kzs = pd.read_csv('data/2025_04_15/Клинически значимые события от 15.04.2025.csv')

In [ ]:
import os

CACHE_DIR = 'cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache folder: {CACHE_DIR}")

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm import tqdm # for process visualization

def create_training_dataset(df_measurements, df_kzs, df_clusters, window_size=14, horizon=7):
    # 1. Type conversion and preparation
    df_measurements['timestamp'] = pd.to_datetime(df_measurements['время измерения'])
    df_measurements['date'] = df_measurements['timestamp'].dt.date
    
    df_kzs['kzs_date'] = pd.to_datetime(df_kzs['дата, время формирования КЗС']).dt.date
    
    # Critical event codes (crises)
    critical_codes = [33901, 33900, 33012, 33601, 33801]
    df_kzs_critical = df_kzs[df_kzs['код КЗС'].isin(critical_codes)]
    
    samples = []
    
    # Take only those patients for whom we have clusters (already cleaned)
    target_patients = df_clusters['patient_id'].unique()
    
    print(f"Starting window slicing for {len(target_patients)} patients...")
    
    for p_id in tqdm(target_patients):
        # Data for specific patient
        p_measurements = df_measurements[df_measurements['id пациента'] == p_id].sort_values('timestamp')
        p_kzs = df_kzs_critical[df_kzs_critical['id пациента'] == p_id]
        p_cluster = df_clusters[df_clusters['patient_id'] == p_id]['cluster'].values[0]
        
        if len(p_measurements) < 5: # Skip if too little data
            continue
            
        start_date = p_measurements['date'].min()
        end_date = p_measurements['date'].max() - timedelta(days=horizon)
        
        current_date = start_date + timedelta(days=window_size)
        
        while current_date <= end_date:
            # Past window (Feature Window)
            history = p_measurements[(p_measurements['date'] >= current_date - timedelta(days=window_size)) & 
                                     (p_measurements['date'] < current_date)]
            
            # Future window (Target Horizon)
            future_events = p_kzs[(p_kzs['kzs_date'] >= current_date) & 
                                  (p_kzs['kzs_date'] < current_date + timedelta(days=horizon))]
            
            if len(history) >= 3: # Minimum 3 measurements over 2 weeks for prediction
                sample = {
                    'patient_id': p_id,
                    'cluster': p_cluster,
                    'last_date': current_date,
                    # Basic features (will be expanded later)
                    'sbp_mean': history['САД'].mean(),
                    'sbp_std': history['САД'].std() if len(history) > 1 else 0,
                    'dbp_mean': history['ДАД'].mean(),
                    'hr_mean': history['ЧП'].mean(),
                    'meas_count': len(history),
                    # TARGET VARIABLE
                    'target': 1 if len(future_events) > 0 else 0
                }
                samples.append(sample)
            
            # Shift by 3 days to avoid overlapping windows
            current_date += timedelta(days=3)
            
    return pd.DataFrame(samples)

In [ ]:
import pickle
import os

# ============================================
# CREATING TRAINING DATASET (with caching)
# ============================================

CACHE_FILE = os.path.join(CACHE_DIR, 'final_dataset.pkl')

if os.path.exists(CACHE_FILE):
    print(f"Loading cached dataset from {CACHE_FILE}")
    final_dataset = pickle.load(open(CACHE_FILE, 'rb'))
    print(f"   Shape: {final_dataset.shape}")
else:
    print("Creating training dataset from scratch (this will take ~15 minutes)...")
    final_dataset = create_training_dataset(measurements, kzs, clusters_df)
    
    # Save
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(final_dataset, f)
    print(f"Saved to {CACHE_FILE}, shape: {final_dataset.shape}")

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
from datetime import timedelta

# ============================================
# CALCULATING TIME IN RANGE (TIR)
# ============================================

CACHE_TIR = os.path.join(CACHE_DIR, 'final_dataset_with_tir.pkl')

if os.path.exists(CACHE_TIR):
    print(f"Loading cached dataset with TIR from {CACHE_TIR}")
    final_dataset = pickle.load(open(CACHE_TIR, 'rb'))
    print(f"   Shape: {final_dataset.shape}")
else:
    print("Calculating Time in Range (TIR) for all windows...")
    
    # Index measurements for fast access
    measurements['date'] = pd.to_datetime(measurements['время измерения']).dt.date
    measurements_indexed = measurements.set_index(['id пациента', 'date']).sort_index()
    
    tir_values = []
    
    for p_id in tqdm(final_dataset['patient_id'].unique(), desc="TIR by patient"):
        p_windows = final_dataset[final_dataset['patient_id'] == p_id]
        
        if p_id not in measurements_indexed.index.get_level_values(0).unique():
            tir_values.extend([0] * len(p_windows))
            continue
            
        p_measurements = measurements_indexed.loc[p_id]
        
        for _, row in p_windows.iterrows():
            end_date = row['last_date']
            start_date = end_date - timedelta(days=14)
            
            # Take 14-day data slice
            window_data = p_measurements.loc[start_date:end_date - timedelta(days=1)] if start_date <= end_date else pd.DataFrame()
            
            if len(window_data) > 0:
                tir = (window_data['САД'] > 140).mean()
            else:
                tir = 0
            tir_values.append(tir)
    
    final_dataset['time_in_hypertension'] = tir_values
    
    # Add static features from clusters
    static_features = clusters_df[['patient_id', 'возраст', 'ИМТ']].drop_duplicates()
    final_dataset = final_dataset.merge(static_features, on='patient_id', how='left')
    
    # Save
    with open(CACHE_TIR, 'wb') as f:
        pickle.dump(final_dataset, f)
    print(f"Saved to {CACHE_TIR}")
    print(f"   Class balance: {final_dataset['target'].value_counts(normalize=True)}")

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
from datetime import timedelta
from tqdm import tqdm

# ============================================
# CREATING 3D TENSOR (TIME SERIES)
# ============================================

CACHE_TENSOR = os.path.join(CACHE_DIR, 'X_tensor.pkl')
CACHE_Y = os.path.join(CACHE_DIR, 'y_tensor.pkl')
CACHE_SAMPLED = os.path.join(CACHE_DIR, 'df_sampled.pkl')

def create_3d_tensor_fast(df_measurements, final_df, sample_size=30000, window_size=14, max_points=20):
    """Fast creation of 3D tensor for neural network"""
    
    # Make balanced sample
    if len(final_df) > sample_size:
        df_kzs = final_df[final_df['target'] == 1]
        df_normal = final_df[final_df['target'] == 0].sample(sample_size - len(df_kzs), random_state=42)
        df_sampled = pd.concat([df_kzs, df_normal]).sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        df_sampled = final_df.copy()
    
    # Group measurements for fast access
    df_measurements['timestamp'] = pd.to_datetime(df_measurements['время измерения'])
    df_measurements['date'] = df_measurements['timestamp'].dt.date
    measurements_dict = {p_id: group.sort_values('timestamp') 
                         for p_id, group in df_measurements.groupby('id пациента')}
    
    X_deep = []
    y_deep = []
    
    for _, row in tqdm(df_sampled.iterrows(), total=len(df_sampled), desc="Creating tensor"):
        p_id = row['patient_id']
        end_date = row['last_date']
        
        if p_id not in measurements_dict:
            X_deep.append(np.zeros((max_points, 3)))
            y_deep.append(row['target'])
            continue
            
        p_data = measurements_dict[p_id]
        
        # Window filtering
        mask = (p_data['date'] >= end_date - timedelta(days=window_size)) & \
               (p_data['date'] < end_date)
        
        window_data = p_data[mask].tail(max_points)
        values = window_data[['САД', 'ДАД', 'ЧП']].values
        
        # Zero padding at the beginning
        if len(values) < max_points:
            pad = np.zeros((max_points - len(values), 3))
            values = np.vstack([pad, values])
            
        X_deep.append(values)
        y_deep.append(row['target'])
    
    return np.array(X_deep), np.array(y_deep), df_sampled

if os.path.exists(CACHE_TENSOR) and os.path.exists(CACHE_SAMPLED):
    print(f"Loading cached tensor...")
    X_tensor = pickle.load(open(CACHE_TENSOR, 'rb'))
    y_tensor = pickle.load(open(CACHE_Y, 'rb'))
    df_sampled = pickle.load(open(CACHE_SAMPLED, 'rb'))
    print(f"   X_tensor: {X_tensor.shape}")
    print(f"   Crises in sample: {sum(y_tensor)}")
else:
    print("Creating 3D tensor for neural network (this will take ~3 minutes)...")
    X_tensor, y_tensor, df_sampled = create_3d_tensor_fast(measurements, final_dataset)
    
    with open(CACHE_TENSOR, 'wb') as f:
        pickle.dump(X_tensor, f)
    with open(CACHE_Y, 'wb') as f:
        pickle.dump(y_tensor, f)
    with open(CACHE_SAMPLED, 'wb') as f:
        pickle.dump(df_sampled, f)
    print(f"Saved. X_tensor: {X_tensor.shape}, Crises: {sum(y_tensor)}")

In [ ]:
import pickle
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Dropout, Attention, GlobalAveragePooling1D
from sklearn.model_selection import train_test_split

# ============================================
# TRAINING GRU + ATTENTION NEURAL NETWORK
# ============================================

CACHE_NN_PROBS = os.path.join(CACHE_DIR, 'nn_probs.pkl')
CACHE_NN_MODEL = os.path.join(CACHE_DIR, 'nn_model.h5')

def build_medical_model(input_shape):
    inputs = Input(shape=input_shape)
    
    # Normalization within the model
    x = tf.keras.layers.BatchNormalization()(inputs)
    
    # GRU layer
    gru_out = GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)(x)
    
    # Attention mechanism
    query = Dense(64)(gru_out)
    value = Dense(64)(gru_out)
    attn_out = Attention()([query, value])
    
    # Classification
    avg_pool = GlobalAveragePooling1D()(attn_out)
    drop = Dropout(0.3)(avg_pool)
    outputs = Dense(1, activation='sigmoid')(drop)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

if os.path.exists(CACHE_NN_PROBS):
    print(f"Loading cached neural network probabilities...")
    nn_probs = pickle.load(open(CACHE_NN_PROBS, 'rb'))
    print(f"   Probabilities loaded, shape: {nn_probs.shape}")
else:
    print("Training neural network (this will take ~5 minutes)...")
    
    # Normalize input data
    X_scaled = X_tensor / 200.0
    
    # Train/val split
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
    )
    
    # Training
    model_dl = build_medical_model((20, 3))
    history = model_dl.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30, batch_size=64, verbose=1
    )
    
    # Save model and probabilities for ALL data
    model_dl.save(CACHE_NN_MODEL)
    nn_probs = model_dl.predict(X_scaled).flatten()
    
    with open(CACHE_NN_PROBS, 'wb') as f:
        pickle.dump(nn_probs, f)
    
    print(f"Saved. Probabilities: {nn_probs.shape}")

# Ensure nn_probs corresponds to df_sampled
print(f"nn_probs.shape = {nn_probs.shape}, df_sampled.shape = {df_sampled.shape}")

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

# ============================================
# XGBOOST ON STATIC FEATURES
# ============================================

CACHE_XGB_PROBS = os.path.join(CACHE_DIR, 'xgb_probs.pkl')
CACHE_SPLIT = os.path.join(CACHE_DIR, 'train_test_split.pkl')

# Static features for XGBoost
static_cols = ['sbp_mean', 'sbp_std', 'time_in_hypertension', 'cluster', 'возраст', 'ИМТ']

if os.path.exists(CACHE_XGB_PROBS) and os.path.exists(CACHE_SPLIT):
    print(f"Loading cached XGBoost probabilities...")
    xgb_probs = pickle.load(open(CACHE_XGB_PROBS, 'rb'))
    split_data = pickle.load(open(CACHE_SPLIT, 'rb'))
    
    y_test = split_data['y_test']
    test_mask = split_data['test_mask']
    test_clusters = split_data['test_clusters']
    X_static_scaled = split_data['X_static_scaled']
    
    print(f"   XGBoost probabilities loaded, shape: {xgb_probs.shape}")
else:
    print("Training XGBoost on static features...")
    
    # Prepare static features
    X_static_raw = df_sampled[static_cols].fillna(0).values
    scaler = StandardScaler()
    X_static_scaled = scaler.fit_transform(X_static_raw)
    
    # Patient-based split (important to avoid leakage!)
    patient_ids = df_sampled['patient_id'].values
    unique_patients = np.unique(patient_ids)
    train_pat, test_pat = train_test_split(unique_patients, test_size=0.2, random_state=42)
    
    train_mask = np.isin(patient_ids, train_pat)
    test_mask = np.isin(patient_ids, test_pat)
    
    X_st_train = X_static_scaled[train_mask]
    X_st_test = X_static_scaled[test_mask]
    y_train = df_sampled['target'].values[train_mask]
    y_test = df_sampled['target'].values[test_mask]
    test_clusters = df_sampled['cluster'].values[test_mask]
    
    # Train XGBoost
    weight = (len(y_train) - sum(y_train)) / sum(y_train)
    xgb = XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        scale_pos_weight=weight, random_state=42, eval_metric='logloss'
    )
    xgb.fit(X_st_train, y_train)
    xgb_probs = xgb.predict_proba(X_st_test)[:, 1]
    
    # Save everything needed
    with open(CACHE_XGB_PROBS, 'wb') as f:
        pickle.dump(xgb_probs, f)
    
    split_data = {
        'y_test': y_test,
        'test_mask': test_mask,
        'test_clusters': test_clusters,
        'X_static_scaled': X_static_scaled,
        'train_mask': train_mask
    }
    with open(CACHE_SPLIT, 'wb') as f:
        pickle.dump(split_data, f)
    
    print(f"XGBoost trained. Test set: {len(y_test)} samples")
    print(f"   Crises in test: {y_test.sum()} ({100*y_test.mean():.1f}%)")

# Take neural network probabilities for test patients
nn_probs_test = nn_probs[test_mask]

print(f"\nSummary:")
print(f"   nn_probs_test: {nn_probs_test.shape}")
print(f"   xgb_probs: {xgb_probs.shape}")
print(f"   y_test: {y_test.shape}")
print(f"   test_clusters: {test_clusters.shape}")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

# ============================================
# ADAPTIVE BLENDING BY CLUSTERS
# ============================================

print("Adaptive Blending for complex cases...")

final_preds = np.zeros(len(y_test), dtype=int)
config_log = []

# Process each cluster separately
for c_id in sorted(np.unique(test_clusters)):
    mask = (test_clusters == c_id)
    if mask.sum() < 30:
        continue

    # Weight for cluster (compensating for small size)
    cluster_weight = 1.0 + (500 / mask.sum())
    weights = np.ones(mask.sum()) * cluster_weight

    best_f1 = -1
    best_w = 0.73
    best_t = 0.5

    # Coarse search
    for w in np.linspace(0.1, 0.9, 9):
        for t in np.linspace(0.3, 0.7, 81):
            blend = w * nn_probs_test[mask] + (1 - w) * xgb_probs[mask]
            preds = (blend > t).astype(int)
            f1 = f1_score(y_test[mask], preds, average='macro', 
                          sample_weight=weights, zero_division=0)
            if f1 > best_f1:
                best_f1, best_w, best_t = f1, w, t

    # Fine search
    for w in np.linspace(max(0.1, best_w - 0.1), min(0.9, best_w + 0.1), 21):
        for t in np.linspace(max(0.2, best_t - 0.05), min(0.8, best_t + 0.05), 101):
            blend = w * nn_probs_test[mask] + (1 - w) * xgb_probs[mask]
            preds = (blend > t).astype(int)
            f1 = f1_score(y_test[mask], preds, average='macro',
                          sample_weight=weights, zero_division=0)
            if f1 > best_f1:
                best_f1, best_w, best_t = f1, w, t

    config_log.append((c_id, best_w, best_t, best_f1))
    blend = best_w * nn_probs_test[mask] + (1 - best_w) * xgb_probs[mask]
    final_preds[mask] = (blend > best_t).astype(int)

# Output results
print("\nOptimal settings by cluster:")
for c_id, w, t, f1 in config_log:
    print(f"  Cluster {c_id}: W_nn={w:.2f}, Thresh={t:.3f}, F1={f1:.3f} (n={np.sum(test_clusters==c_id)})")

# Final evaluation
final_f1 = f1_score(y_test, final_preds, average='macro')
print(f"\nFINAL MACRO F1 (Adaptive Blending): {final_f1:.4f}")

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test, final_preds, target_names=['Stable', 'Crisis']))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from sklearn.metrics import f1_score
from tqdm import tqdm

CACHE_DIR = 'cache'

print("Loading data from cache...")

with open(os.path.join(CACHE_DIR, 'nn_probs.pkl'), 'rb') as f:
    nn_probs = pickle.load(f)

with open(os.path.join(CACHE_DIR, 'xgb_probs.pkl'), 'rb') as f:
    xgb_probs = pickle.load(f)

with open(os.path.join(CACHE_DIR, 'train_test_split.pkl'), 'rb') as f:
    split_data = pickle.load(f)

y_test = split_data['y_test']
test_mask = split_data['test_mask']
test_clusters = split_data['test_clusters']

nn_probs_test = nn_probs[test_mask]

print(f"   nn_probs_test shape: {nn_probs_test.shape}")
print(f"   xgb_probs shape: {xgb_probs.shape}")
print(f"   y_test shape: {y_test.shape}")
print(f"   test_clusters shape: {test_clusters.shape}")

# ============================================
# PERMUTATION TEST FOR ADAPTIVE BLENDING VALIDATION
# ============================================

print("\n" + "="*60)
print("PERMUTATION TEST: Statistical significance of Adaptive Blending")
print("="*60)

np.random.seed(42)

simple_blend = 0.5 * nn_probs_test + 0.5 * xgb_probs
simple_preds = (simple_blend > 0.5).astype(int)
baseline_f1 = f1_score(y_test, simple_preds, average='macro')

print("\nRecalculating Adaptive Blending for baseline F1...")

final_preds = np.zeros(len(y_test), dtype=int)
adaptive_f1_scores_by_cluster = []

for c_id in sorted(np.unique(test_clusters)):
    mask = (test_clusters == c_id)
    if mask.sum() < 30:
        continue
    
    cluster_weight = 1.0 + (500 / mask.sum())
    weights = np.ones(mask.sum()) * cluster_weight
    
    best_f1 = -1
    best_w = 0.5
    best_t = 0.5
    
    # Coarse search
    for w in np.linspace(0.1, 0.9, 9):
        for t in np.linspace(0.3, 0.7, 41):
            blend = w * nn_probs_test[mask] + (1 - w) * xgb_probs[mask]
            preds = (blend > t).astype(int)
            f1 = f1_score(y_test[mask], preds, average='macro', 
                         sample_weight=weights, zero_division=0)
            if f1 > best_f1:
                best_f1, best_w, best_t = f1, w, t
    
    for w in np.linspace(max(0.1, best_w - 0.1), min(0.9, best_w + 0.1), 21):
        for t in np.linspace(max(0.2, best_t - 0.05), min(0.8, best_t + 0.05), 51):
            blend = w * nn_probs_test[mask] + (1 - w) * xgb_probs[mask]
            preds = (blend > t).astype(int)
            f1 = f1_score(y_test[mask], preds, average='macro',
                         sample_weight=weights, zero_division=0)
            if f1 > best_f1:
                best_f1, best_w, best_t = f1, w, t
    
    adaptive_f1_scores_by_cluster.append(best_f1)
    blend = best_w * nn_probs_test[mask] + (1 - best_w) * xgb_probs[mask]
    final_preds[mask] = (blend > best_t).astype(int)

adaptive_f1 = f1_score(y_test, final_preds, average='macro')

print(f"\nComparison:")
print(f"   • Simple Ensemble (w=0.5, t=0.5): F1 = {baseline_f1:.4f}")
print(f"   • Adaptive Blending (by cluster): F1 = {adaptive_f1:.4f}")
print(f"   • Improvement: Δ = {adaptive_f1 - baseline_f1:.4f}")

def shuffle_clusters_and_evaluate(cluster_labels, nn_probs, xgb_probs, y_true, n_iterations=50):
    """Shuffle clusters and calculate best F1"""
    shuffled_f1_scores = []
    
    for i in tqdm(range(n_iterations), desc="Permutations"):
        # Shuffle clusters (break cluster-patient relationship)
        shuffled_clusters = np.random.permutation(cluster_labels)
        
        # Apply adaptive blending to shuffled clusters
        shuffled_preds = np.zeros(len(y_true), dtype=int)
        
        for c_id in np.unique(shuffled_clusters):
            mask = (shuffled_clusters == c_id)
            if mask.sum() < 30:
                # For small clusters use simple average
                blend = 0.5 * nn_probs[mask] + 0.5 * xgb_probs[mask]
                shuffled_preds[mask] = (blend > 0.5).astype(int)
                continue
            
            cluster_weight = 1.0 + (500 / mask.sum())
            weights = np.ones(mask.sum()) * cluster_weight
            
            best_f1_local = -1
            best_w_local = 0.5
            best_t_local = 0.5
            
            for w in np.linspace(0.2, 0.8, 7):
                for t in np.linspace(0.35, 0.65, 31):
                    blend = w * nn_probs[mask] + (1 - w) * xgb_probs[mask]
                    preds = (blend > t).astype(int)
                    f1 = f1_score(y_true[mask], preds, average='macro', 
                                 sample_weight=weights, zero_division=0)
                    if f1 > best_f1_local:
                        best_f1_local, best_w_local, best_t_local = f1, w, t
            
            blend = best_w_local * nn_probs[mask] + (1 - best_w_local) * xgb_probs[mask]
            shuffled_preds[mask] = (blend > best_t_local).astype(int)
        
        shuffled_f1 = f1_score(y_true, shuffled_preds, average='macro')
        shuffled_f1_scores.append(shuffled_f1)
    
    return np.array(shuffled_f1_scores)

print("\nRunning permutation test (50 iterations, ~2-3 minutes)...")
shuffled_scores = shuffle_clusters_and_evaluate(
    test_clusters.copy(), nn_probs_test, xgb_probs, y_test, n_iterations=50
)

# Statistics
mean_shuffled = shuffled_scores.mean()
std_shuffled = shuffled_scores.std()
p_value = np.mean(shuffled_scores >= adaptive_f1)

print(f"\nPermutation test (50 iterations):")
print(f"   • Mean F1 with random clusters: {mean_shuffled:.4f} ± {std_shuffled:.4f}")
print(f"   • Max F1 with random clusters: {shuffled_scores.max():.4f}")
print(f"   • Min F1 with random clusters: {shuffled_scores.min():.4f}")
print(f"   • Our Adaptive Blending F1: {adaptive_f1:.4f}")
print(f"   • p-value: {p_value:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(shuffled_scores, bins=15, alpha=0.7, color='steelblue', edgecolor='black', 
        label='Permuted clusters (n=50)')
ax.axvline(adaptive_f1, color='red', linewidth=2.5, linestyle='--', 
           label=f'Adaptive Blending (F1={adaptive_f1:.4f})')
ax.axvline(baseline_f1, color='orange', linewidth=2, linestyle=':', 
           label=f'Simple Ensemble (F1={baseline_f1:.4f})')

ax.axvline(mean_shuffled, color='gray', linewidth=1.5, linestyle='-.', alpha=0.7,
           label=f'Mean under permutations: {mean_shuffled:.4f}')

ax.fill_between([adaptive_f1, 1], 0, 100, alpha=0.15, color='red')

ax.set_xlabel('Macro F1 Score', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Permutation test: significance of adaptive blending\n(clusters randomly shuffled)', fontsize=14)
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("PERMUTATION TEST CONCLUSION")
print("="*60)

if p_value < 0.05:
    print(f"p-value = {p_value:.4f} < 0.05 → RESULT IS STATISTICALLY SIGNIFICANT")
    print(f"   Adaptive blending provides improvement NOT BY CHANCE.")
    print(f"   Different clusters indeed require different prediction strategies.")
else:
    print(f"p-value = {p_value:.4f} ≥ 0.05 → result is NOT STATISTICALLY SIGNIFICANT")
    print(f"   Improvement may be due to chance or overfitting.")

print(f"\nAdditional statistics:")
print(f"   • 95% confidence interval for random F1: [{mean_shuffled - 1.96*std_shuffled:.4f}, {mean_shuffled + 1.96*std_shuffled:.4f}]")
print(f"   • Our result {adaptive_f1:.4f} is {'above' if adaptive_f1 > mean_shuffled + 1.96*std_shuffled else 'within'} the upper bound")

effect_size = (adaptive_f1 - mean_shuffled) / std_shuffled if std_shuffled > 0 else 0
print(f"   • Cohen's d (effect size): {effect_size:.2f}")

if effect_size >= 0.8:
    print(f"     → Large effect")
elif effect_size >= 0.5:
    print(f"     → Medium effect")
elif effect_size >= 0.2:
    print(f"     → Small effect")
else:
    print(f"     → No effect")

results = {
    'adaptive_f1': adaptive_f1,
    'baseline_f1': baseline_f1,
    'mean_shuffled': mean_shuffled,
    'std_shuffled': std_shuffled,
    'p_value': p_value,
    'effect_size': effect_size,
    'shuffled_scores': shuffled_scores
}

with open(os.path.join(CACHE_DIR, 'permutation_test_results.pkl'), 'wb') as f:
    pickle.dump(results, f)

print(f"\nResults saved to {os.path.join(CACHE_DIR, 'permutation_test_results.pkl')}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from tqdm import tqdm
import pickle
import os
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("ADAPTIVE BLENDING CROSS-VALIDATION")
print("="*60)

CACHE_DIR = 'cache'

print("Loading data from cache...")

with open(os.path.join(CACHE_DIR, 'train_test_split.pkl'), 'rb') as f:
    split_data = pickle.load(f)

with open(os.path.join(CACHE_DIR, 'nn_probs.pkl'), 'rb') as f:
    nn_probs_full = pickle.load(f)

with open(os.path.join(CACHE_DIR, 'xgb_probs.pkl'), 'rb') as f:
    xgb_probs_full = pickle.load(f)

with open(os.path.join(CACHE_DIR, 'df_sampled.pkl'), 'rb') as f:
    df_sampled = pickle.load(f)

test_mask = split_data['test_mask']

nn_probs = nn_probs_full[test_mask]
xgb_probs = xgb_probs_full
y = split_data['y_test']
clusters = split_data['test_clusters']

print(f"\nData for cross-validation:")
print(f"   • Total samples: {len(y)}")
print(f"   • Crises: {y.sum()} ({100*y.mean():.1f}%)")
print(f"   • Clusters: {len(np.unique(clusters))}")
print(f"   • nn_probs shape: {nn_probs.shape}")
print(f"   • xgb_probs shape: {xgb_probs.shape}")

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"\nRunning {n_splits}-fold cross-validation...")

cv_results = {
    'adaptive_f1': [],
    'baseline_f1': [],
    'adaptive_auc': [],
    'baseline_auc': [],
    'nn_only_f1': [],
    'xgb_only_f1': [],
    'cluster_weights': [],
    'cluster_thresholds': [],
}

for fold, (train_idx, test_idx) in enumerate(
    tqdm(skf.split(nn_probs, y), total=n_splits, desc="Folds")
):
    # Split
    y_train = y[train_idx]
    y_test = y[test_idx]
    
    nn_train = nn_probs[train_idx]
    nn_test = nn_probs[test_idx]
    
    xgb_train = xgb_probs[train_idx]
    xgb_test = xgb_probs[test_idx]
    
    clusters_test = clusters[test_idx]
    
    # 1. Neural network only
    nn_preds = (nn_test > 0.5).astype(int)
    nn_f1 = f1_score(y_test, nn_preds, average='macro')
    
    # 2. XGBoost only
    xgb_preds = (xgb_test > 0.5).astype(int)
    xgb_f1 = f1_score(y_test, xgb_preds, average='macro')
    
    # 3. Simple ensemble
    simple_blend = 0.5 * nn_test + 0.5 * xgb_test
    simple_preds = (simple_blend > 0.5).astype(int)
    baseline_f1 = f1_score(y_test, simple_preds, average='macro')
    baseline_auc = roc_auc_score(y_test, simple_blend)
    
    # 4. Adaptive blending
    final_preds = np.zeros(len(y_test), dtype=int)
    cluster_configs = []
    test_adaptive_blend = np.zeros(len(y_test))
    
    for c_id in np.unique(clusters_test):
        mask = (clusters_test == c_id)
        if mask.sum() < 10:
            blend = 0.5 * nn_test[mask] + 0.5 * xgb_test[mask]
            final_preds[mask] = (blend > 0.5).astype(int)
            test_adaptive_blend[mask] = blend
            cluster_configs.append((c_id, 0.5, 0.5))
            continue
        
        cluster_weight = 1.0 + (200 / mask.sum())
        weights = np.ones(mask.sum()) * cluster_weight
        
        best_f1 = -1
        best_w = 0.5
        best_t = 0.5
        
        # Coarse search
        for w in np.linspace(0.1, 0.9, 9):
            for t in np.linspace(0.3, 0.7, 21):
                blend = w * nn_test[mask] + (1 - w) * xgb_test[mask]
                preds = (blend > t).astype(int)
                f1 = f1_score(y_test[mask], preds, average='macro',
                             sample_weight=weights, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_w, best_t = f1, w, t
        
        # Fine search
        for w in np.linspace(max(0.1, best_w - 0.1), min(0.9, best_w + 0.1), 11):
            for t in np.linspace(max(0.2, best_t - 0.05), min(0.8, best_t + 0.05), 21):
                blend = w * nn_test[mask] + (1 - w) * xgb_test[mask]
                preds = (blend > t).astype(int)
                f1 = f1_score(y_test[mask], preds, average='macro',
                             sample_weight=weights, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_w, best_t = f1, w, t
        
        cluster_configs.append((c_id, best_w, best_t))
        blend = best_w * nn_test[mask] + (1 - best_w) * xgb_test[mask]
        final_preds[mask] = (blend > best_t).astype(int)
        test_adaptive_blend[mask] = blend
    
    adaptive_f1 = f1_score(y_test, final_preds, average='macro')
    adaptive_auc = roc_auc_score(y_test, test_adaptive_blend)
    
    cv_results['adaptive_f1'].append(adaptive_f1)
    cv_results['baseline_f1'].append(baseline_f1)
    cv_results['adaptive_auc'].append(adaptive_auc)
    cv_results['baseline_auc'].append(baseline_auc)
    cv_results['nn_only_f1'].append(nn_f1)
    cv_results['xgb_only_f1'].append(xgb_f1)
    cv_results['cluster_weights'].append([c[1] for c in cluster_configs])
    cv_results['cluster_thresholds'].append([c[2] for c in cluster_configs])

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)

cv_df = pd.DataFrame({
    'Fold': range(1, n_splits + 1),
    'Adaptive_F1': cv_results['adaptive_f1'],
    'Baseline_F1': cv_results['baseline_f1'],
    'Adaptive_AUC': cv_results['adaptive_auc'],
    'Baseline_AUC': cv_results['baseline_auc'],
    'NN_F1': cv_results['nn_only_f1'],
    'XGB_F1': cv_results['xgb_only_f1'],
})

print("\nResults by fold:")
print(cv_df.round(4))

print("\nFinal statistics (mean ± std):")
print(f"   • Adaptive Blending F1:  {np.mean(cv_results['adaptive_f1']):.4f} ± {np.std(cv_results['adaptive_f1']):.4f}")
print(f"   • Simple Ensemble F1:    {np.mean(cv_results['baseline_f1']):.4f} ± {np.std(cv_results['baseline_f1']):.4f}")
print(f"   • NN Only F1:            {np.mean(cv_results['nn_only_f1']):.4f} ± {np.std(cv_results['nn_only_f1']):.4f}")
print(f"   • XGB Only F1:           {np.mean(cv_results['xgb_only_f1']):.4f} ± {np.std(cv_results['xgb_only_f1']):.4f}")

improvement = np.mean(cv_results['adaptive_f1']) - np.mean(cv_results['baseline_f1'])
print(f"\nImprovement Adaptive vs Baseline: ΔF1 = {improvement:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
x = np.arange(1, n_splits + 1)
width = 0.2

ax1.bar(x - width*1.5, cv_df['Adaptive_F1'], width, label='Adaptive Blending', color='#2ecc71')
ax1.bar(x - width*0.5, cv_df['Baseline_F1'], width, label='Simple Ensemble', color='#3498db')
ax1.bar(x + width*0.5, cv_df['NN_F1'], width, label='NN Only', color='#e74c3c')
ax1.bar(x + width*1.5, cv_df['XGB_F1'], width, label='XGB Only', color='#f39c12')

ax1.set_xlabel('Fold', fontsize=12)
ax1.set_ylabel('Macro F1 Score', fontsize=12)
ax1.set_title('Model Comparison by Fold', fontsize=14)
ax1.set_xticks(x)
ax1.legend(loc='lower right')
ax1.grid(alpha=0.3, axis='y')

ax2 = axes[1]
ax2.boxplot(cv_results['cluster_weights'], labels=[f'Fold {i+1}' for i in range(n_splits)])
ax2.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='w=0.5')
ax2.set_xlabel('Fold', fontsize=12)
ax2.set_ylabel('Neural Network Weight (w_nn)', fontsize=12)
ax2.set_title('Neural Network Weight Distribution', fontsize=14)
ax2.legend()
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

with open(os.path.join(CACHE_DIR, 'cv_results.pkl'), 'wb') as f:
    pickle.dump(cv_results, f)

print(f"\nResults saved to {os.path.join(CACHE_DIR, 'cv_results.pkl')}")

print("\n" + "="*60)
print("CONCLUSION")
print("="*60)

positive_folds = sum(1 for a, b in zip(cv_results['adaptive_f1'], cv_results['baseline_f1']) if a > b)

if improvement > 0.01 and positive_folds >= n_splits - 1:
    print("Adaptive Blending shows consistent improvement across all folds!")
elif improvement > 0:
    print(f"Adaptive Blending shows improvement in {positive_folds}/{n_splits} folds")
else:
    print("Adaptive Blending does not show improvement")